# TP2

## Extraire les mots clés de tous les documemts CAMille de 1920 avec Yake

### Imports

In [ ]:
import os
import yake

In [ ]:
# Instantier l'extracteur de mots clés
kw_extractor = yake.KeywordExtractor(lan="fr", top=50)
kw_extractor

https://github.com/INESCTEC/yake

In [ ]:
# Choisir une année
year = 1920 # année des jeux olympiques à Anvers

In [ ]:
# Lister les fichiers de cette année
data_path = '../../data'
txt_path = '../../data/txt'
txts = [f for f in os.listdir(txt_path) if os.path.isfile(os.path.join(txt_path, f)) and str(year) in f]
len(txts)

In [ ]:
# Stocker le contenu de ces fichiers dans une liste
content_list = []
for txt in txts:
    with open(os.path.join(txt_path, txt), 'r', encoding='utf-8') as f:
        content_list.append(f.read())

In [ ]:
# Compter le nombre d'éléments (=fichiers) dans la liste
len(content_list)

In [ ]:
for f in sorted(txts)[:10]:
    text = open(os.path.join(txt_path, f), 'r', encoding="utf-8").read()
    keywords = kw_extractor.extract_keywords(text)
    kept = []
    for kw, score in keywords:
        words = kw.split()
        if len(words) >= 2:
            kept.append(kw)
    print(f"{f} mentions these keywords: {', '.join(kept)}...")

## Enrechir la liste de stopwords

### Import

In [ ]:
from collections import Counter
from wordcloud import WordCloud
import os
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from IPython.display import Image

In [ ]:
# Stopwords (Idem que dans s1)
sw = stopwords.words("french")
sw += ["les", "plus", "cette", "fait", "faire", "être", "deux", "comme", "dont", "tout",
       "ils", "bien", "sans", "peut", "tous", "après", "ainsi", "donc", "cet", "sous",
       "celle", "entre", "encore", "toutes", "pendant", "moins", "dire", "cela", "non",
       "faut", "trois", "aussi", "dit", "avoir", "doit", "contre", "depuis", "autres",
       "van", "het", "autre", "jusqu", "ville", "rossel", "dem", 
       "celui", "elles", "ceux", "chez", "leurs", "vers", "déjà", "très", "quelques", "hier", "toute", "devant", "Devant", "DEVANT", "plusieurs",
       "trop", "rien", "dès", "tant", "assez", "alors", "car", "générale", "quand", "dernier", "fit", # pronoms etc.
       "etc", "ETC", "Mme", "MME", "référ", "div", "réf", "serv", "adr", "Aujourd", "aujourd", "AUJOURD", "toujours",  # abbréviations communes
       "cap", "trav", "com", "con", "ord", "pet", "dés", "ecr", "écrire", "prix", "aven", "rue", "adresser", "cherche", 
       "soir", "vendre", "prés", "ans", "mod", "Brux", "BRUX", "demande", "vend", "agence", "maison", "francs", "louer", "Mén", "MÉN", "sér",
       "vente", "avenue", "bonne", "neuf", "nouvelle", "beau", "bon", "place", "heures", "grand", "mois", "jour", "jeune", 
       "grande", "bons", "vente", "avant", "belle", "peu", "société", "commerce", "fille", "occasion", "lieu", "suite",
       "courant", "gros", "près", "petit", "neuve", "heure", "bel", "sér", "dame", "boulevard", "chaussée", "divers", "grands",
       "tél", "TÈL", "Tèl", "téléphone", "meubles", "meublés", "prendre", "cuisine", "jard", "vue", "vis", "pers", "PERS", "Pers", 
       "nom", "bonnes", "petite", "chambre", "occas", "coucher", "offre", "jardin", "garni", "gaz", "femme", "vient",
       "seul", "mètres", "chêne", "quart", "piano", "bas", "maison", "hôtel", "ici", "Saint", "hom", "ménage", "chamb", "démandé", "env", "matin", 
       "cent", "dix", "six", "cinq", "écrit", "cont", "désire", "café", "dos", "ordre", "notaire", "notaires", "bureaux", "bureau", "saint", "part", "jours",
       "cours", "mot", "salle", "homme", "achat", "pris", "villa", "vieux", "rentier", "salon", "fort", "Jean",
       "haut", "service", "Service", "SERVICE", "hui", "HUI", "demi", "présenter", "vendra", "franc", "Franc", "voir", "main",
       "gare", "cuis", "mal", "quartier", "nouveau", "dom", "demander", "bail", "offre", "ventes", # abbréviations et mots présent en grand nombre en raison des petites annonces, nom du journal etc.
       "rua", "rie", "ruo", "uno", "aveo", "ecrire", "ire", "bur", "eor", "rte", "ter", "Franco", "franco", # mots non bien captés par l'OCR
       "Bruxelles", "BRUXELLES", "bruxelles", "brux", "BRUX", "Anvers", "ANVERS", "anvers", "Namur", "MIDI", "Midi", "midi", 
       "Nord", "NORD", "nord", "Paris", "paris", "PARIS", "Louvain", "LOUVAIN", "Liège",
       "lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche", # jours de la semaine
       "janvier", "février", "mars", "avril", "mai", "juin", "juillet", "août", "septembre", "octobre", "novembre", "décembre"] # mois de l'année  
sw = set(sw)


In [ ]:
# Ecrire tout le contenu dans un fichier temporaire
temp_path = '../../data/tmp'
if not os.path.exists(temp_path):
    os.mkdir(temp_path)
with open(os.path.join(temp_path, f'{year}.txt'), 'w', encoding='utf-8') as f:
    f.write(' '.join(content_list))

In [ ]:
# Imprimer le contenu du fichier et constater les "déchets"
with open(os.path.join(temp_path, f'{year}.txt'), 'r', encoding='utf-8') as f:
    before = f.read()

before[:500]

In [ ]:
def clean_text(year, folder=None):
    if folder is None:
        input_path = f"{year}.txt"
        output_path = f"{year}_clean.txt"
    else:
        input_path = f"{folder}/{year}.txt"
        output_path = f"{folder}/{year}_clean.txt"
    output = open(output_path, "w", encoding='utf-8')
    with open(input_path, 'r', encoding='utf-8') as f:
        text = f.read()
        words = nltk.wordpunct_tokenize(text)
        kept = [w.upper() for w in words if len(w) > 2 and w.isalpha() and w.lower() not in sw]
        kept_string = " ".join(kept)
        output.write(kept_string)
    return f'Output has been written in {output_path}!'

In [ ]:
clean_text(year, folder=temp_path)

In [ ]:
# Vérifier le résultat
with open(os.path.join(temp_path, f'{year}_clean.txt'), 'r', encoding='utf-8') as f:
    after = f.read()

after[:500]

## Nuage de mots

### Afficher les termes les plus fréquents


In [ ]:
frequencies = Counter(after.split())
print(frequencies.most_common(100))

### Créer, stocker et afficher le nuage de mots

In [ ]:
cloud = WordCloud(width=2000, height=1000, background_color='white').generate_from_frequencies(frequencies)
cloud.to_file(os.path.join(temp_path, f"{year}.png"))
Image(filename=os.path.join(temp_path, f"{year}.png"))

## Reconnaissance d'entités nommées avec SpaCy

### Imports

In [ ]:
from collections import defaultdict
import spacy

### Charger SpaCy

In [ ]:
nlp = spacy.load('fr_core_news_md')

### Charger les texts de 1920

In [ ]:
# Charger le texte
n=1000000
text = open(os.path.join(temp_path, f'{year}.txt'), encoding='utf-8').read()[:n] # adapté pour prendre le fichier tmp de l'année choisie plus haut



In [ ]:
%%time
# Traiter le texte

doc = nlp(text)

### Trouver les personnes

In [ ]:
# Compter les entités
people = defaultdict(int)
for ent in doc.ents:
    if ent.label_ == ("PER") and len(ent.text) > 3:
        people[ent.text] += 1

In [ ]:
# Trier et imprimer

sorted_people = sorted(people.items(), key=lambda kv: kv[1], reverse=True)

for person, freq in sorted_people[:50]:
    print(f"{person} apparait {freq} fois dans le corpus")

### Trouver les organisations

In [ ]:
# Compter les entités
organisations = defaultdict(int)
for ent in doc.ents:
    if ent.label_ == ("ORG") and len(ent.text) > 3:
        organisations[ent.text] += 1

In [ ]:
# Trier et imprimer

sorted_organisations = sorted(organisations.items(), key=lambda kv: kv[1], reverse=True)

for organisation, freq in sorted_organisations[:50]:
    print(f"{organisation} apparait {freq} fois dans le corpus")

### Trouver les lieux

In [ ]:
# Compter les entités
locations = defaultdict(int)
for ent in doc.ents:
    if ent.label_ == ("LOC") and len(ent.text) > 3:
        locations[ent.text] += 1

In [ ]:
# Trier et imprimer

sorted_locations = sorted(locations.items(), key=lambda kv: kv[1], reverse=True)

for location, freq in sorted_locations[:50]:
    print(f"{location} apparait {freq} fois dans le corpus")

## Sentiment analysis 

### Imports

In [ ]:
from textblob import Blobber
from textblob_fr import PatternTagger, PatternAnalyzer
import pandas as pd # pour créer le tableau
from pandas import option_context # pour pouvoir afficher les phrases complètes dans le tableau


### Création de deux fonctions 'get_subjectivity' et 'get_positivity'

In [ ]:
tb = Blobber(pos_tagger=PatternTagger(), analyzer=PatternAnalyzer())
Valsubjectivity = [] # pour stocker le résultat de la fonction

def get_subjectivity(input_text):
    blob = tb(input_text)
    polarity, subjectivity = blob.sentiment
    subjectivity_perc = f"{100*subjectivity:.0f}"
    if subjectivity > 0:
        subjectivity_str = f"{subjectivity_perc}% subjective"
    else:
        subjectivity_str = "perfectly objective"
    Valsubjectivity.append(f"{subjectivity_str}")

In [ ]:
tb = Blobber(pos_tagger=PatternTagger(), analyzer=PatternAnalyzer())
Valpolarity = []

def get_polarity(input_text):
    blob = tb(input_text)
    polarity, subjectivity = blob.sentiment
    polarity_perc = f"{100*abs(polarity):.0f}"
    if polarity > 0:
        polarity_str = f"{polarity_perc}% positive"
    elif polarity < 0:
        polarity_str = f"{polarity_perc}% negative"
    else:
        polarity_str = "neutral"
    Valpolarity.append(f"{polarity_str}")

### Analyser le sentiment des phrases

In [ ]:
# stocker les 10 phrases
sentences = ["Un bon de visite sera remis aux amateurs qui pourront se rendre sur place pour y examiner les quantités disponibles.", 
"Notre triomphe est louchant, réconfortant, et tout patriote sensible doit s’en réjouir.", 
"C’est une question que nous devons laisser à dés spécialistes le soin d’élucider.", 
"Une affluence considérable d’étrangers peut donc exercer une heureuse influence sur le cours du change, étant donné que celui-ci est déterminé par la grande loi de l’offre et de la demande.", 
"MM. Delacroix et Jaspar ont sans doute compris l'urgence du contrôle institué de la sorte et nous feront connaître sans tarder la forme du registre exigé, afin de ne pas reculer d’un instant l'entrée en vigueur du nouvel arrêté.", 
"C’est une honte éternelle pour la marine britannique.", 
"Aussi expira-t-il dans le plus noble et le plus beau des combats.", 
"Un conseil de cabinet se tiendra jeudi matin, à 9 heures, au ministère des finances.", 
"Mais la catastrophe est venue, et avec elle s’est effondrée la meilleure excuse de cette hégémonie de Berlin, que la Kölnische Volkszeitung appelle aujourd’hui « la capitale des provinces barbares de l’Est de l’Elbe.", 
"La doctrine de Marx fait l’objet d’une chaude discussion."]

In [ ]:
# calculer la polarité et la subjectivité pour toutes les phrases. 
# Les résultats sont stockés dans Valpolarity et Valsubjectivity.
for x in sentences: 
    get_polarity(x)
    get_subjectivity(x)  

In [ ]:
# Tableaux des résultats
d = {"phrase": sentences, "polarity": Valpolarity, "subjectivity": Valsubjectivity}
df = pd.DataFrame(data=d)
with option_context('display.max_colwidth', 400): # permet d'afficher les phrases complètes
    display(df)